<a href="https://colab.research.google.com/github/brenoslivio/SummerSchool_BashGit/blob/main/Git.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unix, Bash & Git: Tools for Reproducible Science

## Git Notebook

*This Jupyter Notebook was developed for the Summer School on Trends in Multi-Omics Data Analysis for Microbial Ecology and Biotechnology (2026) to be presented on 16.07.2026.*

*For any questions regarding the material provided here, please contact:*

*Breno L. S. de Almeida: brenoslivio@usp.br*

---

### Table of contents

- [A note on running this notebook](#note)
- [1. Installing and configuring Git](#sec1)
- [2. Creating & connecting a GitHub repository](#sec2)
    - [2.1 Creating a repository on GitHub](#sec2-1)
    - [2.2 Generating & adding an SSH key](#sec2-2)
    - [2.3 Cloning your repository](#sec2-3)
    - [2.4 Signing commits with SSH (optional)](#sec2-4)
- [3. Everyday workflow: edit, add, commit, push, pull](#sec3)
    - [3.1 Editing files](#sec3-1)
    - [3.2 Staging & committing](#sec3-2)
    - [3.3 Pushing & pulling](#sec3-3)
- [4. Branches](#sec4)
    - [4.1 Creating and switching branches](#sec4-1)
    - [4.2 Writing a script in your branch](#sec4-2)
    - [4.3 Committing your branch and merging back into main](#sec4-3)
- [5. .gitignore](#sec5)
- [6. Reverting changes](#sec6)
    - [6.1 Viewing history with git log](#sec6-1)
    - [6.2 Safe revert with git revert](#sec6-2)
    - [6.3 Hard reset (dangerous)](#sec6-3)
- [7. Capstone project: your first pull request](#sec7)

<a name="note"></a>

### A note on running this notebook

This notebook builds directly on the **Bash notebook** — if you haven't
gone through it yet, do that first, since we reuse the same
notebook-friendly habits established there:

- `nano` and other interactive editors **will not work** inside a Colab
  code cell. Wherever a Git workflow would normally use `nano <file>`, we
  instead write files in one go with a heredoc
  (`cat > file << 'EOF' ... EOF`, see section **2.10** of the Bash
  notebook), or you can create/edit the file by hand using Colab's file
  browser (folder icon on the left sidebar).
- Every code cell runs in its **own, fresh subprocess**. Nothing you
  `export`, `eval`, or `cd` into in one cell carries over to the next —
  you'll need to `cd` into your repository again (and restart the SSH
  agent) at the start of most cells below.

That second point matters a lot for Git, because talking to GitHub over
SSH requires a **running `ssh-agent` with your key loaded in memory**.
On Google Colab, the `%%bash` cell magic does **not** reliably keep that
agent alive for the rest of the cell, so instead we chain everything
into a **single `!` shell command** with `&&`, using `\` to spread it
over multiple lines for readability:

```bash
!eval "$(ssh-agent -s)" && \
  ssh-add ~/.ssh/id_ed25519 && \
  ssh -T git@github.com
```

You'll see this `!... && \` pattern repeated throughout the notebook
whenever a cell talks to the remote repository (`clone`, `push`, `pull`,
`fetch`) — it's not a copy-paste mistake, it's required every single
time on Colab. Cells that only touch your local files (creating
branches, writing files, committing) can still use `%%bash` normally. On
your own machine, in a real terminal, you'd only need to start the agent
once per session.

<a name="sec1"></a>

### 1. Installing and configuring Git

- `apt install git` — installs Git (already present on most systems,
  including Colab, but it doesn't hurt to check)
- `git config --global user.name "<Your Name>"` — the name attached to
  every commit you make
- `git config --global user.email "<you@example.com>"` — the email
  attached to every commit (use the same one as your GitHub account)
- `git config --list` — show your current configuration

In [1]:
!apt update # fetch updates

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [100 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.5 MB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,312 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,089 kB]
Get:

In [2]:
!apt install git # install git, if not already present

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 101 not upgraded.


Configure your identity — replace the example name and email below with
your own:

In [3]:
%%bash

git config --global user.name "Linus Torvalds"
git config --global user.email torvalds@osdl.org

**📝 Exercise 1** — Replace the example name/email above with your own,
re-run the cell, then confirm the change with `git config --list`.

In [4]:
!git config --list

filter.lfs.required=true
filter.lfs.clean=git-lfs clean -- %f
filter.lfs.smudge=git-lfs smudge -- %f
filter.lfs.process=git-lfs filter-process
user.name=Linus Torvalds
user.email=torvalds@osdl.org


<a name="sec2"></a>

### 2. Creating & connecting a GitHub repository

<a name="sec2-1"></a>

#### 2.1 Creating a repository on GitHub

Create a GitHub account if you don't already have one, then create your
first repository:

1. Once logged in, go to **Your repositories** and click the green
   **New** button.
2. Give the repository a name and a short description.
3. Select **Private**, and check **Add a README file** (leave the rest
   as default).
4. Click **Create repository**.

You now have your first remote repository, living on GitHub's servers.

<a name="sec2-2"></a>

#### 2.2 Generating & adding an SSH key

- `ssh-keygen -t ed25519 -C "<comment>"` — generate a new SSH key pair
  (a public and a private key)
- `ssh-agent -s` — start an SSH agent that holds your decrypted key in
  memory
- `ssh-add <path/to/private_key>` — load your private key into the
  running agent
- `ssh -T git@github.com` — test the connection to GitHub over SSH

We're going to connect to GitHub using **SSH keys** rather than a
password — see [About remote
repositories](https://docs.github.com/en/get-started/getting-started-with-git/about-remote-repositories)
and [About SSH](https://docs.github.com/en/authentication/connecting-to-github-with-ssh/about-ssh)
for background.

Generate a new key pair (press Enter to accept the default file
location, and set a passphrase or leave it empty):

In [5]:
!ssh-keygen -t ed25519 -C "your_email@example.com"

Generating public/private ed25519 key pair.
Enter file in which to save the key (/root/.ssh/id_ed25519): 
Created directory '/root/.ssh'.
Enter passphrase (empty for no passphrase): test
Enter same passphrase again: test
Your identification has been saved in /root/.ssh/id_ed25519
Your public key has been saved in /root/.ssh/id_ed25519.pub
The key fingerprint is:
SHA256:W3I/dI4kWpAlEJtiO1c+7klATEOAFfh+rPk779q/3+s your_email@example.com
The key's randomart image is:
+--[ED25519 256]--+
|   +++*o. .      |
|  o  o + +       |
|   .o = +        |
|   ..+ o .       |
|   .o.o S = o .  |
|    .ooo O = +   |
|     +  =   + .  |
|    o .+ .   o   |
|     .+**.oo..E. |
+----[SHA256]-----+


Copy your **public** key into your GitHub account under
**Settings → SSH and GPG keys → New SSH key**. Print it with:

In [6]:
!cat ~/.ssh/id_ed25519.pub

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAII5JL07b8HsG+wHmLmornlZZt4cNN3dBu6ql3gwv3tBW your_email@example.com


As explained in the note above, starting the agent and adding your key
only counts for the single cell it runs in — so from here on, every
cell that talks to GitHub starts with the same two lines:

In [7]:
!eval "$(ssh-agent -s)" && \
  ssh-add ~/.ssh/id_ed25519 && \
  ssh -T git@github.com

Agent pid 1745
Enter passphrase for /root/.ssh/id_ed25519: test
Identity added: /root/.ssh/id_ed25519 (your_email@example.com)
Host key verification failed.


You should see a message like `Hi <username>! You've successfully
authenticated, but GitHub does not provide shell access.` — that confirms
the key was accepted.

<a name="sec2-3"></a>

#### 2.3 Cloning your repository

- `git clone <url>` — download a full copy of a remote repository,
  including its entire history

In [8]:
!eval "$(ssh-agent -s)" && \
  ssh-add ~/.ssh/id_ed25519 && \
  git clone git@github.com:<username>/<name_of_repository>.git

Agent pid 1833
Enter passphrase for /root/.ssh/id_ed25519: test
Identity added: /root/.ssh/id_ed25519 (your_email@example.com)
/bin/bash: line 1: username: No such file or directory


Move into the cloned repository and confirm everything is in order:

In [ ]:
%%bash

cd <name_of_repository>
git status

Should give you the following output:

```
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
```

Every code cell from here on assumes you're inside
`<name_of_repository>` — since each cell starts fresh (just like
`~/bash_practice` in the Bash notebook), begin each one with
`cd <name_of_repository>` (using your own repository's name).

<a name="sec2-4"></a>

#### 2.4 Signing commits with SSH (optional)

You can sign your commits with your SSH key, proving to others that a
commit really came from you. Check for existing SSH keys in your GitHub
account and, this time, add your public key a **second** time as a
**Signing Key** (not an Authentication Key).

- `git config --global gpg.format ssh` — tell Git to use SSH (instead of
  GPG) for signing
- `git config --global user.signingkey <path/to/public_key>` — the key
  Git should sign commits with

In [ ]:
!git config --global gpg.format ssh

Set your SSH signing key, substituting the path with your own public
key:

In [ ]:
!git config --global user.signingkey /PATH/TO/.SSH/KEY.PUB

This is enough to have GitHub show your commits as **Verified**. If you
also want commits verified *locally* (e.g. by teammates cloning your
repo), you additionally need an [allowed signers
file](https://docs.github.com/en/authentication/managing-commit-signature-verification/about-commit-signature-verification).

Let's check what our configuration looks like now:

In [ ]:
!git config --list

<a name="sec3"></a>

### 3. Everyday workflow: edit, add, commit, push, pull

<a name="sec3-1"></a>

#### 3.1 Editing files

Recall from the Bash notebook (**2.10**) that `nano` doesn't work inside
a notebook cell. To edit `README.md`, we write it in one go with a
heredoc instead. Here's some example Markdown syntax you can reuse for
your own README files:

In [ ]:
%%bash

cd <name_of_repository>
cat > README.md << 'EOF'
# My Summer School Project

## About

This README was written directly from a notebook cell using a heredoc,
without ever opening an interactive text editor.

### Sections

#### Subsections

This is how we make numbered lists:

1. Using
2. Git
3. Is
4. Useful

This is how we make bullet points:

- and
- can
- be
- applicable
- in
- biology

This is how we hyperlink inside the text, e.g. back to the
[Sections](#sections) heading above.

You can also link to external websites, like the [class
repository](https://github.com/brenoslivio/SummerSchool_BashGit).

You can also make tables:

|Column 1|Column 2|Column 3|Column 4|
|:-----:|:--:|:-----:|:----:|
|just|cause|I|can|
|but|so|can|you|

We can embed figures like this:

![Figure 1 - Name of figure](relative/location/of/the/figure/figure1.png)

And show code blocks, e.g. a snippet from the Bash notebook:

```bash
cd ~/bash_practice
find . -name "*.txt" -exec realpath {} +
```
EOF
cat README.md

<a name="sec3-2"></a>

#### 3.2 Staging & committing

- `git status` — see what has changed
- `git add <file>` — stage a file for the next commit (`git add .`
  stages everything in the current directory)
- `git commit -m "<message>"` — record a snapshot of the staged
  changes

In [ ]:
%%bash

cd <name_of_repository>
git status
git add README.md
git commit -m "Update README with project description"

<a name="sec3-3"></a>

#### 3.3 Pushing & pulling

- `git push origin main` — upload your local commits to the remote
  repository
- `git pull origin main` — download and merge changes from the remote
  repository (always do this before pushing, to avoid conflicts)

As always, talking to the remote means starting the SSH agent again in
the same cell:

In [ ]:
!cd <name_of_repository> && \
  eval "$(ssh-agent -s)" && \
  ssh-add ~/.ssh/id_ed25519 && \
  git pull origin main && \
  git push origin main

Now go check your repository on GitHub — your new README should be
live. 🎉

**📝 Exercise 3** — Make one more small change to `README.md` (e.g. add
your name and research topic under a new `## About me` section), then
stage, commit, pull, and push it, following the same pattern as
above.

<a name="sec4"></a>

### 4. Branches

One of Git's most useful features is branching: you can "copy" the
current state of your project into a separate line of work, make changes
there without touching `main`, and merge them back once you're happy
with the result — all without interfering with anyone else using the
repository in the meantime.

<a name="sec4-1"></a>

#### 4.1 Creating and switching branches

- `git branch` — list branches, and show which one you're currently on
- `git branch <name>` — create a new branch (without switching to it)
- `git switch <name>` — switch to an existing branch (`git checkout
  <name>` also works)
- `git switch -c <name>` — create **and** switch to a new branch in one
  step

In [ ]:
%%bash

cd <name_of_repository>
git branch
git switch -c scripts-branch
git branch

<a name="sec4-2"></a>

#### 4.2 Writing a script in your branch

Now let's do some work on this branch. Instead of a long, unrelated
script, we'll write something in the same style as the shell script we
built in section **2.10** of the Bash notebook — short, and tied to the
same sample-QC scenario from that notebook's capstone:

In [ ]:
%%bash

cd <name_of_repository>
mkdir -p scripts
cat > scripts/qc_summary.sh << 'EOF'
#!/bin/bash
data_dir=${1:-data}
total=$(find "$data_dir" -name "*.txt" 2>/dev/null | wc -l)
passed=$(grep -rl "status: passed" "$data_dir" 2>/dev/null | wc -l)
failed=$(grep -rl "status: failed" "$data_dir" 2>/dev/null | wc -l)
echo "Total sample files: $total"
echo "Passed: $passed"
echo "Failed: $failed"
EOF
chmod +x scripts/qc_summary.sh
ls -l scripts/

This script expects a folder of `.txt` sample files formatted like
`status: passed` / `status: failed` — exactly like the ones you built in
the Bash notebook's capstone project. Let's give it something to
summarize and run it:

In [ ]:
%%bash

cd <name_of_repository>
mkdir -p data
echo "status: passed" > data/sample_S1.txt
echo "status: failed" > data/sample_S2.txt
echo "status: passed" > data/sample_S3.txt
./scripts/qc_summary.sh

**📝 Exercise 4.1** — Add two more sample files to `data/` (any status),
then re-run `./scripts/qc_summary.sh` and confirm the counts update
correctly.

<a name="sec4-3"></a>

#### 4.3 Committing your branch and merging back into main

- `git add -A` — stage all changes (new, modified, and deleted files)
- `git merge <branch>` — merge another branch's history into the branch
  you're currently on

Commit the script and sample data on your branch, and push the branch
itself (not `main`) to GitHub:

In [ ]:
!cd <name_of_repository> && \
  git add -A && \
  git commit -m "Add QC summary script and sample data" && \
  eval "$(ssh-agent -s)" && \
  ssh-add ~/.ssh/id_ed25519 && \
  git push origin scripts-branch

Now let's bring those changes into `main`. Switch back, make sure `main`
is up to date, then merge:

In [ ]:
!cd <name_of_repository> && \
  git switch main && \
  eval "$(ssh-agent -s)" && \
  ssh-add ~/.ssh/id_ed25519 && \
  git pull origin main && \
  git merge scripts-branch -m "Merge scripts-branch into main" && \
  git push origin main

`main` on GitHub now includes everything you built on `scripts-branch`.
This branch → merge → push cycle is exactly how you'll work on real
projects: keep `main` stable, do your work on a branch, and merge only
once it's ready.

<a name="sec5"></a>

### 5. .gitignore

Not everything in a project should be tracked by Git — large files,
secrets, or generated output don't belong in your history. A
`.gitignore` file tells Git which paths to skip.

- `*.ext` — ignore every file with that extension
- `!important.ext` — an exception: track this specific file even though
  a broader pattern above would ignore it
- `folder/` — ignore an entire folder
- lines starting with `#` are comments

In [ ]:
%%bash

cd <name_of_repository>
cat > .gitignore << 'EOF'
# Ignore all .pkl files
*.pkl

# Except this one
!important_results.pkl

# Ignore all zipped files
*.zip

# Ignore a whole folder
raw_downloads/
EOF
cat .gitignore

**📝 Exercise 5** — Create a throwaway file that matches one of the
patterns above (e.g. `touch scratch.pkl`), run `git status`, and confirm
Git does **not** offer to track it. Then stage, commit and push the
`.gitignore` file itself.

In [ ]:
!cd <name_of_repository> && \
  touch scratch.pkl && \
  git status && \
  git add .gitignore && \
  git commit -m "Add .gitignore" && \
  eval "$(ssh-agent -s)" && \
  ssh-add ~/.ssh/id_ed25519 && \
  git pull origin main && \
  git push origin main

<a name="sec6"></a>

### 6. Reverting changes

Sooner or later you'll commit (and maybe push) something that breaks
your code. Git gives you a few ways to undo that, ranging from safe to
dangerous.

<a name="sec6-1"></a>

#### 6.1 Viewing history with git log

- `git log` — show the commit history (hash, author, date, message)
- `git log --oneline` — a compact, one-line-per-commit view

Let's deliberately introduce a bug, so we have something to revert —
here we overwrite `qc_summary.sh` with a broken version that drops the
pass/fail counts:

In [ ]:
%%bash

cd <name_of_repository>
cat > scripts/qc_summary.sh << 'EOF'
#!/bin/bash
data_dir=${1:-data}
total=$(find "$data_dir" -name "*.txt" 2>/dev/null | wc -l)
echo "Total sample files: $total"
EOF
git add scripts/qc_summary.sh
git commit -m "Oops: accidentally removed the pass/fail counts"
./scripts/qc_summary.sh
git log --oneline

<a name="sec6-2"></a>

#### 6.2 Safe revert with git revert

- `git revert <commit>` — create a **new** commit that undoes the
  changes from `<commit>`, keeping full history intact

In [ ]:
%%bash

cd <name_of_repository>
git log --oneline
git revert --no-edit HEAD
./scripts/qc_summary.sh
git log --oneline

`git revert` is safe to use even after pushing, because it never
rewrites history — it just adds a new commit on top. Push it like any
other commit:

In [ ]:
!cd <name_of_repository> && \
  eval "$(ssh-agent -s)" && \
  ssh-add ~/.ssh/id_ed25519 && \
  git push origin main

<a name="sec6-3"></a>

#### 6.3 Hard reset (dangerous)

> **Warning:** `git reset --hard` rewrites history and **permanently
> discards** every commit after the one you reset to, along with any
> uncommitted changes. Only use it on a branch nobody else relies on —
> never on a shared `main` unless you are certain.

- `git reset --hard <commit>` — move the current branch's pointer to
  `<commit>` and discard everything after it
- `git push --force` — required afterwards, since the rewritten history
  no longer matches the remote

To keep this safe, we'll demonstrate it on `scripts-branch` rather than
`main`:

In [ ]:
%%bash

cd <name_of_repository>
git switch scripts-branch
git log --oneline

Copy a commit hash from the log above — the one **before** the "Oops"
commit — and substitute `<hash-of-earlier-commit>` below:

In [ ]:
!cd <name_of_repository> && \
  git reset --hard <hash-of-earlier-commit> && \
  eval "$(ssh-agent -s)" && \
  ssh-add ~/.ssh/id_ed25519 && \
  git push origin scripts-branch --force

<a name="sec7"></a>

### 7. Capstone project: your first pull request

Time to make a real, public contribution using the exact same skills:
forking, branching, committing, and pushing. Scenario:

> The class keeps a shared roster of participants in the repository
> [`brenoslivio/SummerSchool_BashGit`](https://github.com/brenoslivio/SummerSchool_BashGit),
> inside a folder called `students/`. Your task is to add a text file
> with your own name to that folder — but since you don't have write
> access to someone else's repository, you'll do it the way almost all
> open-source contributions happen: **fork it, make your change, and
> open a pull request.**

1. **Fork the repository.** Go to
   https://github.com/brenoslivio/SummerSchool_BashGit and click **Fork**
   (top right) to create your own copy of it under your GitHub account.

2. **Clone your fork** (not the original!) via SSH — note the URL below
   uses *your* username, not the instructor's:

In [ ]:
!eval "$(ssh-agent -s)" && \
  ssh-add ~/.ssh/id_ed25519 && \
  git clone git@github.com:<your_username>/SummerSchool_BashGit.git

3. **Create a branch for your change.** Never commit directly to `main`,
   even on your own fork — it keeps things tidy if you want to contribute
   again later:

In [ ]:
%%bash

cd SummerSchool_BashGit
git switch -c add-<your_name>

4. **Add your file.** Create `students/<your_name>.txt` containing your
   name, the same way we've been writing files throughout this notebook:

In [ ]:
%%bash

cd SummerSchool_BashGit
mkdir -p students
cat > students/<your_name>.txt << 'EOF'
<Your Full Name>
EOF
cat students/<your_name>.txt

5. **Stage, commit, and push** the new file to **your fork** — the
   remote is still called `origin` and still points at your fork, not
   the instructor's repository:

In [ ]:
!cd SummerSchool_BashGit && \
  git add students/<your_name>.txt && \
  git commit -m "Add <your_name> to students" && \
  eval "$(ssh-agent -s)" && \
  ssh-add ~/.ssh/id_ed25519 && \
  git push origin add-<your_name>

6. **Open the pull request.** Go back to your fork on GitHub — you
   should see a banner offering to **Compare & pull request** for the
   branch you just pushed. Click it, and double-check:
   - the **base repository** is `brenoslivio/SummerSchool_BashGit`, base
     branch `main`
   - the **head repository** is your fork, branch `add-<your_name>`

   Add a short title/description, then click **Create pull request**.
7. That's it! Your instructor will review and merge it. If they request
   changes, just commit again on the same branch and push — the pull
   request updates automatically, no need to open a new one.